# Multimodal_Cartera_Fondos

## Primera etapa: de la voz del usuario a texto

Este notebook inicia una aplicación para crear una cartera de fondos a partir de lo que expresa un usuario. Implementa **captura de voz → preparación del audio → Whisper → transcripción → revisión del usuario**.

Se basa en `Code-Multimodales-Clase/Code/7. Audio_transcription_with_Whisper.ipynb`: conserva Whisper Base, PyTorch, AutoModelForSpeechSeq2Seq, AutoProcessor y pipeline de Transformers. Añade captura por micrófono y tratamiento explícito de frecuencia de muestreo y canales.

**Alcance:** esta etapa obtiene texto; todavía no interpreta el perfil inversor ni selecciona fondos. No se deducen preferencias que el usuario no haya expresado.

Ejemplo de petición: «Quiero invertir 10.000 euros durante cinco años, aportar 200 euros al mes y asumir un riesgo moderado. Prefiero fondos diversificados y con comisiones bajas».


## 1. Preparación

Ejecuta las celdas en orden en Jupyter local. La grabación usa el micrófono del equipo donde se ejecuta el kernel y requiere permiso del sistema operativo. En Colab o un servidor remoto, utiliza la alternativa de cargar un WAV en el entorno del notebook.

La primera carga descarga el modelo de Hugging Face y necesita Internet. Después puede reutilizarse la caché. La inferencia se ejecuta en CPU o GPU local. Los audios y textos se guardan localmente en `datos_cartera`; utiliza datos ficticios para la demo y evita incluir información personal en GitHub.


In [ ]:
%pip install torch transformers accelerate safetensors numpy scipy sounddevice


In [ ]:
from pathlib import Path
from math import gcd
import json
from datetime import datetime, timezone

import numpy as np
import torch
from scipy.io import wavfile
from scipy.signal import resample_poly
from IPython.display import Audio, display
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

DIRECTORIO_DATOS = Path("datos_cartera")
DIRECTORIO_DATOS.mkdir(exist_ok=True)
RUTA_AUDIO = DIRECTORIO_DATOS / "peticion_usuario.wav"
FRECUENCIA_GRABACION = 16000
DURACION_SEGUNDOS = 20


## 2. Capturar la voz

Ejecuta la celda siguiente para grabar durante 20 segundos. Puedes cambiar `DURACION_SEGUNDOS`. La captura comienza al ejecutar la celda.

Si no hay micrófono o el kernel es remoto, omite la grabación y ejecuta la celda de carga alternativa. Si falla la captura, comprueba los permisos del micrófono y la entrada de audio seleccionada en el sistema.


In [ ]:
def grabar_peticion(ruta, duracion=20, frecuencia=16000):
    import sounddevice as sd

    if not 1 <= duracion <= 120:
        raise ValueError("La duración debe estar entre 1 y 120 segundos.")
    print(f"Habla ahora: grabación de {duracion} segundos.")
    try:
        audio = sd.rec(
            int(duracion * frecuencia), samplerate=frecuencia,
            channels=1, dtype="float32"
        )
        sd.wait()
    except Exception as exc:
        raise RuntimeError(
            "No se pudo grabar. Comprueba el micrófono y sus permisos "
            "o utiliza un archivo WAV con la carga alternativa."
        ) from exc
    wavfile.write(str(ruta), frecuencia, audio)
    print(f"Grabación guardada en: {ruta.resolve()}")
    return ruta

RUTA_AUDIO = grabar_peticion(
    RUTA_AUDIO, DURACION_SEGUNDOS, FRECUENCIA_GRABACION
)


### Alternativa: utilizar un WAV existente

Ejecuta esta celda **solo si utilizas un archivo en lugar de grabar**. Cambia la ruta por la de tu audio. Se admite WAV PCM o flotante; si tienes MP3 o M4A, conviértelo antes a WAV.


In [ ]:
# Descomenta y ajusta estas líneas para utilizar un WAV existente:
# RUTA_AUDIO = Path("mi_peticion.wav")
# if not RUTA_AUDIO.is_file():
#     raise FileNotFoundError(f"No existe el audio: {RUTA_AUDIO.resolve()}")


In [ ]:
if not RUTA_AUDIO.is_file():
    raise FileNotFoundError("Graba una petición o configura la ruta de un WAV existente.")
display(Audio(filename=str(RUTA_AUDIO)))


## 3. Preparar la señal

El ejemplo de clase divide por 32768 para audio PCM de 16 bits. Aquí se adapta la normalización al tipo real del WAV, se convierte estéreo a mono y se remuestrea a 16 kHz. Así Whisper recibe una señal con su frecuencia de muestreo explícita.


In [ ]:
def preparar_audio(ruta, frecuencia_objetivo=16000):
    frecuencia, datos = wavfile.read(str(ruta))
    if frecuencia <= 0 or datos.size == 0:
        raise ValueError("El archivo de audio está vacío o su frecuencia es inválida.")
    tipo = datos.dtype
    if np.issubdtype(tipo, np.unsignedinteger):
        centro = (np.iinfo(tipo).max + 1) / 2
        audio = (datos.astype(np.float32) - centro) / centro
    elif np.issubdtype(tipo, np.signedinteger):
        limites = np.iinfo(tipo)
        escala = max(abs(int(limites.min)), int(limites.max))
        audio = datos.astype(np.float32) / escala
    elif np.issubdtype(tipo, np.floating):
        audio = datos.astype(np.float32)
    else:
        raise ValueError(f"Tipo de audio no compatible: {tipo}")
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    elif audio.ndim != 1:
        raise ValueError("Se esperaba audio mono o multicanal.")
    if not np.isfinite(audio).all():
        raise ValueError("El audio contiene valores no válidos.")
    audio = np.clip(audio, -1.0, 1.0)
    if np.max(np.abs(audio)) < 1e-5:
        raise ValueError("El audio parece contener silencio. Revisa el micrófono.")
    if frecuencia != frecuencia_objetivo:
        divisor = gcd(int(frecuencia), frecuencia_objetivo)
        audio = resample_poly(
            audio, frecuencia_objetivo // divisor, int(frecuencia) // divisor
        ).astype(np.float32)
    return np.ascontiguousarray(audio, dtype=np.float32), frecuencia_objetivo

audio_preparado, frecuencia = preparar_audio(RUTA_AUDIO)
print(f"Frecuencia: {frecuencia} Hz")
print(f"Duración: {len(audio_preparado) / frecuencia:.1f} segundos")
print(f"Forma: {audio_preparado.shape}; tipo: {audio_preparado.dtype}")


## 4. Cargar Whisper como en el ejemplo de clase

Se reutilizan el modelo y el procesador en una única pipeline de reconocimiento de voz. Whisper Base es el punto de partida de clase; su transcripción puede equivocarse en cantidades o nombres de fondos, por lo que se revisará el resultado antes de continuar.


In [ ]:
MODELO_ASR = "openai/whisper-base"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODELO_ASR, torch_dtype=torch_dtype,
    low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(MODELO_ASR)

pipe = pipeline(
    task="automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)
print(f"Whisper preparado en {device}")


## 5. Convertir voz a texto

Se indica español y la tarea de transcripción para conservar el idioma. La segmentación permite procesar grabaciones de más de 30 segundos.


In [ ]:
result = pipe(
    {"array": audio_preparado, "sampling_rate": frecuencia},
    chunk_length_s=30,
    generate_kwargs={"language": "spanish", "task": "transcribe"},
)
texto_transcrito = result["text"].strip()
if not texto_transcrito:
    raise ValueError("No se obtuvo texto. Repite la grabación con voz más clara.")
print("Petición transcrita:")
print(texto_transcrito)


## 6. Revisar y guardar la petición

Escucha el audio y revisa especialmente importes, plazo, aportaciones y preferencias. Introduce una corrección si es necesario. Confirma explícitamente antes de guardar el texto que recibirá la siguiente etapa.


In [ ]:
print("Texto detectado:", texto_transcrito)
correccion = input("Texto corregido (Enter para conservar la transcripción): ").strip()
texto_revisado = correccion or texto_transcrito
confirmacion = input("¿Confirmas este texto? Escribe SI para guardarlo: ").strip().casefold()

if confirmacion in {"si", "sí"}:
    peticion_usuario = {
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "archivo_audio": str(RUTA_AUDIO.resolve()),
        "modelo_asr": MODELO_ASR,
        "idioma": "es",
        "texto_transcrito": texto_transcrito,
        "texto_confirmado": texto_revisado,
        "confirmado_por_usuario": True,
    }
    ruta_peticion = DIRECTORIO_DATOS / "peticion_usuario.json"
    ruta_peticion.write_text(
        json.dumps(peticion_usuario, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Petición confirmada guardada en: {ruta_peticion.resolve()}")
else:
    peticion_usuario = None
    print("Petición pendiente de revisión. Vuelve a ejecutar esta celda para confirmar.")


## 7. Comprobación manual y continuación

Prueba la grabación con estas peticiones y compara el audio con el texto:

- «Quiero invertir diez mil euros durante cinco años y aportar doscientos euros al mes».
- «Prefiero fondos de renta fija y necesito disponer del dinero dentro de dos años».
- «No sé cuánto riesgo quiero asumir; necesito que me preguntes antes de proponer una cartera».

Comprueba que las cantidades y las negaciones se conservan. Prueba también un WAV estéreo o de otra frecuencia para verificar el preprocesado. Whisper puede producir texto incorrecto ante ruido o silencio; la revisión humana sigue siendo necesaria.

**Siguiente etapa del proyecto:** interpretar `texto_confirmado`, preguntar por los datos que falten, consultar un catálogo de fondos y construir una propuesta con cálculos verificables. Esa etapa aún no está implementada en este notebook.
